In [ ]:
%load_ext autoreload
%autoreload 2

In [41]:
import logging

import matplotlib.pyplot as plt  # type: ignore
import numpy as np
import healpy as hp

from mlpng import Core, get_itotcov_ell
from mlpng.utils import (
    setup_logging,
    plot_cl_alm,
    plot_predictions,
    plot_histogram,
    plot_elsner_comp,
    pol_str,
)
from mlpng.generator import generate_alm, generate_alm_nl, lens_alms
from mlpng.utils.utils import print_errors

logger = setup_logging(__name__, level=logging.DEBUG)

mpi_comm = None

In [ ]:
core = Core(
    [
        "settings/elsner.json",
        "--nsims",
        "30",
        "--narray",
        "1",
        # "--lmax",
        # "1024",
        "--fnl_range",
        "-100",
        "100",
        "--pols",
        "T",
    ]
)

In [ ]:
core.init_estimator(verbose=True)

In [44]:
elsner_cl_files = "data/elsner/cl_wmap5_bao_sn.dat"
data = np.loadtxt(elsner_cl_files)

# want to add the mono and dipole terms to match other code
data = np.vstack([[0, 0, 0, 0], [1, 0, 0, 0], data])

ells = data[:, 0]
c_ell_tt = data[:, 1]
c_ell_ee = data[:, 2]
c_ell_te = data[:, 3]
c_ells = np.array([c_ell_tt, c_ell_ee, np.zeros_like(c_ell_tt), c_ell_te])

# scale = ells * (ells + 1) / (2 * np.pi)
# c_ells *= scale
c_ells *= (core.cosmo.camb_params.TCMB * 1e6) ** 2

In [45]:
pols = core.pol_idxs()
core.c_ell = c_ells[:, : core.nell]

core.cov = cov = core.b_ell**2 * core.c_ell + core.n_ell
inoise = np.full(core.n_ell.shape, 1e-16)
inoise[pols, core.lmin :] = 1 / core.n_ell[pols, core.lmin :]

icov = np.zeros_like(cov)
icov[pols, core.lmin :] = 1 / cov[pols, core.lmin :]
icov = get_itotcov_ell(icov, inoise, core.b_ell)
icov = icov[pols, pols]
core.icov = np.array(icov[:, : core.nell])

This code generates the alms

$$a_{\ell m} = a_{\ell m}^{{G}} + f_{NL}^X a_{\ell m}^{NG}$$
with
$$a_{\ell m}^{NG,loc'} = \int dr r^2 \left[ \alpha_\ell(r)\left(\int d^2 \hat{n} Y_{\ell m}^\star (\hat{n}) B(r,\hat{n})^2 \right)\right]$$
and
$$\alpha_\ell(r)=\frac{2}{\pi} \int_0^\infty dk k^2 \Delta_\ell^T(k) j_\ell(k r)$$
$$\beta_\ell(r)=\frac{2}{\pi} \int_0^\infty dk k^{-1} \Delta_\phi \Delta_\ell^T(k) j_\ell(k r)$$
$$B(r, \hat{n}) = \sum_{\ell,m} \frac{\beta_\ell (r)}{C_\ell} a_{\ell m} Y_{\ell m}$$
where $\Delta_\phi$ is primordial normalization, $\Delta_\ell^T(k)$ is the transfer function, $j_\ell(k r)$ are the spherical bessel functions   

In [ ]:
alm_l = generate_alm(core)
alm_ng = generate_alm_nl(core, alm_l)

In [47]:
fnls = core.rng.uniform(core.fnl_min, core.fnl_max, (core.nsims, core.ndups, 1, 1))
alms = alm_l[:, None, core.pol_idxs()] + fnls * alm_ng[:, None]

In [ ]:
plot_cl_alm(
    core,
    alm_l[0, 0],
    title="linear alms",
    # labels=plt_labels,
    plot_camb=True,
    plot_noise=False,
    plot_full_camb=True,
    show=True,
)

plot_cl_alm(
    core,
    alm_ng[0],
    title="non-linear alms",
    plot_camb=False,
    plot_noise=False,
    plot_full_camb=False,
    show=True,
)

In [ ]:
plot_cl_alm(
    core,
    alms[0, 0],
    title="alms final",
    plot_camb=True,
    plot_noise=False,
    plot_full_camb=True,
    show=True,
)

In [ ]:
sim = 0  # core.rng.integers(core.nsims)
eidx = 1  # core.rng.integers(1, 1001)
plot_elsner_comp(
    core,
    alm_l[sim, core.pol_idxs()],
    alm_ng[sim],
    index=1,
    show=True,
    elsner_pols=core.pol_idxs(),
)
plot_elsner_comp(
    core,
    alm_l[sim, core.pol_idxs()],
    alm_ng[sim],
    index=eidx,
    show=True,
    plot_func=plt.plot,
    elsner_pols=core.pol_idxs(),
)
plot_elsner_comp(
    core,
    alm_l[sim, core.pol_idxs()],
    alm_ng[sim],
    index=eidx,
    show=True,
    elsner_pols=core.pol_idxs(),
    plot_func=plt.loglog,
)

In [ ]:
alm_l_avg = np.mean(alm_l[:, core.pol_idxs()], axis=0)
alm_ng_avg = np.mean(alm_ng, axis=0)

plot_elsner_comp(
    core,
    alm_l_avg,
    alm_ng_avg,
    average=25,
    title="avged comp",
    show=True,
    plot_func=plt.semilogy,
    elsner_pols=core.pol_idxs(),
)

## Lensing and Patching

In [ ]:
alm_lensed, phi_map = lens_alms(core, alms)

In [ ]:
sim = 0
for i, pol in enumerate(core.pol_idxs()):
    pstr = pol_str(pol)

    ylabel = r"$\ell(\ell+1)/2\pi\;C_{\ell}" + f"^{pstr}$"
    plot_cl_alm(
        core,
        alm_lensed[sim, i],
        # save_file=filebase,
        ylabel=ylabel,
        plot_camb=True,
        plot_noise=False,
        plot_full_camb=True,
        show=True,
        lmin=2,
    )

## Estimator

In [ ]:
pols = core.pol_idxs()

c_ell = core.c_ell  # if not lensed else core.c_ell_lensed
# cov = c_ell  # core.b_ell**2 * c_ell + core.n_ell
cov = c_ell + core.n_ell / core.b_ell**2

ib = np.ones_like(c_ell)
ib[pols, core.lmin :] = 1 / core.b_ell[pols, core.lmin :]
# n_ell = core.n_ell[pols]
# cov = c_ell + ib * n_ell * ib

ic_ell = np.zeros_like(cov)
ic_ell[pols, core.lmin :] = 1 / c_ell[pols, core.lmin :]
ic_ell = ic_ell[pols]

fisher = core.estimator.compute_fisher_isotropic(ic_ell, comm=None)
print(f"Fisher: {fisher}, standard deviation: {1 / np.sqrt(fisher)}")

In [ ]:
fisher = core.estimator.compute_fisher_isotropic(core.icov, comm=mpi_comm)
print(f"Fisher: {fisher}, standard deviation: {1 / np.sqrt(fisher)}")

In [ ]:
theta_batch = int(np.floor(1.5 * core.lmax + 1)) // core.n_cpus


def alm_loader(i):
    # if core.lensing:
    #     return alm_lensed[i, 0]
    # else:
    return alms[i, 0]


def icov_func(alm, icov):
    ret = np.zeros_like(alm)
    for pol in range(alm.shape[0]):
        ic = icov[pol, pol] if icov.ndim == 3 else icov[pol]
        ret[pol] = hp.almxfl(alm[pol], ic)
    # ret[ret == 0] = 1e16
    return ret


idxs = range(core.nsims)

# C_ell here should be the C^TT_ell + N_ell, total signal + noise power spectrum per smith and zald
cov = core.b_ell**2 * core.c_ell + core.n_ell
# cov = core.c_ell + core.n_ell / core.b_ell**2
icov = np.zeros_like(core.c_ell)
icov[..., core.lmin :] = 1 / (cov)[..., core.lmin :]
# icov *= core.b_ell**2

estimates, _, _, _ = core.estimator.compute_estimate_batch(
    lambda a: icov_func(alm_loader(a), icov),
    # lambda a: alm_loader(a),
    idxs,
    theta_batch=theta_batch,
    fisher=fisher,
    lin_term=0,
)

In [ ]:
print("fnl shape", fnls.shape, "estimates shape", estimates.shape, "fisher", fisher)
fnls_flat = fnls[:, 0, 0, 0]
print_errors(fnls_flat, estimates, fisher)
plot_predictions(fnls_flat, estimates, fisher=fisher, show=True)
plot_histogram(fnls_flat, estimates, show=True)